## True Wait-K Architecture

In [56]:
import torch
from torch import nn
from transformers import (
    MarianTokenizer,
    MarianConfig,
    MarianMTModel,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset, DatasetDict
import pandas as pd
from tqdm.auto import tqdm

In [23]:
import torch
from torch import nn
from transformers import (
    MarianTokenizer,
    MarianConfig,
    MarianMTModel,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from transformers.models.marian.modeling_marian import (
    MarianDecoderLayer,
    MarianAttention
)
from datasets import Dataset, DatasetDict
import pandas as pd
from tqdm.auto import tqdm
from functools import wraps


def create_waitk_forward(k_value):
    """Create a wait-k forward function for cross-attention."""
    def waitk_forward(
        self,
        hidden_states,
        key_value_states=None,
        past_key_value=None,
        attention_mask=None,
        layer_head_mask=None,
        output_attentions=False,
    ):
        # Check if this is cross-attention
        is_cross_attention = key_value_states is not None
        
        if is_cross_attention:
            # This is cross-attention - apply wait-k masking
            query_states = self.q_proj(hidden_states)
            key_states = self.k_proj(key_value_states)
            value_states = self.v_proj(key_value_states)
            
            batch_size, tgt_len, _ = query_states.size()  # [B, T, H]
            _, src_len, _ = key_states.size()  # [B, S, H]
            
            # Reshape for multi-head attention
            query_states = query_states.view(batch_size, tgt_len, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, T, d]
            key_states = key_states.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, S, d]
            value_states = value_states.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)  # [B, H, S, d]
            
            # Compute attention scores
            attn_weights = torch.matmul(query_states, key_states.transpose(-2, -1)) / (self.head_dim ** 0.5)  # [B, H, T, S]
            
            # Apply wait-k mask
            waitk_mask = torch.full((batch_size, self.num_heads, tgt_len, src_len), float('-inf'), 
                                    device=attn_weights.device, dtype=attn_weights.dtype)
            for t in range(tgt_len):
                allowed = min(src_len, t + k_value)
                if allowed > 0:
                    waitk_mask[:, :, t, :allowed] = 0.0
            
            attn_weights = attn_weights + waitk_mask
            
            # Apply encoder attention mask if provided
            if attention_mask is not None:
                if attention_mask.dim() == 2:
                    attention_mask = attention_mask.unsqueeze(1).unsqueeze(1)  # [B, 1, 1, S]
                attn_weights = attn_weights.masked_fill(~attention_mask.bool(), float('-inf'))
            
            # Apply softmax
            attn_weights = nn.functional.softmax(attn_weights, dim=-1)
            
            if layer_head_mask is not None:
                attn_weights = attn_weights * layer_head_mask
            
            attn_weights = nn.functional.dropout(attn_weights, p=self.dropout, training=self.training)
            
            attn_output = torch.matmul(attn_weights, value_states)  # [B, H, T, d]
            attn_output = attn_output.transpose(1, 2).contiguous()  # [B, T, H, d]
            attn_output = attn_output.view(batch_size, tgt_len, self.embed_dim)  # [B, T, H]
            attn_output = self.out_proj(attn_output)
            
            return attn_output, attn_weights if output_attentions else None, past_key_value
        else:
            # Self-attention - use parent implementation
            return MarianAttention.forward(
                self,
                hidden_states=hidden_states,
                key_value_states=key_value_states,
                past_key_value=past_key_value,
                attention_mask=attention_mask,
                layer_head_mask=layer_head_mask,
                output_attentions=output_attentions,
            )
    
    return waitk_forward


class WaitKDecoderLayer(MarianDecoderLayer):
    """Decoder layer with wait-k cross-attention."""
    
    def __init__(self, config, k=3):
        super().__init__(config)
        # Monkey-patch the encoder_attn.forward method to apply wait-k
        original_forward = self.encoder_attn.forward
        # Monkey-patch the encoder_attn.forward method to apply wait-k
        # Store original forward and create bound method
        waitk_forward_func = create_waitk_forward(k)
        # Bind the function to the encoder_attn instance
        import types
        self.encoder_attn.forward = types.MethodType(waitk_forward_func, self.encoder_attn)


class MarianWaitK(MarianMTModel):
    k: int  # Type annotation for the attribute
    
    def __init__(self, config, k: int = 3):
        super().__init__(config)
        object.__setattr__(self, 'k', k)
        
        # Replace decoder layers with wait-k versions
        self.model.decoder.layers = nn.ModuleList([
            WaitKDecoderLayer(config, k=k) for _ in range(config.decoder_layers)
        ])

    def copy_decoder_weights_from(self, source_model):
        """Copy weights from source model's decoder layers."""
        if not isinstance(source_model, MarianMTModel):
            return
        
        source_layers = source_model.model.decoder.layers
        target_layers = self.model.decoder.layers
        
        for src_layer, tgt_layer in zip(source_layers, target_layers):
            # Copy all weights (encoder_attn weights will be copied, forward is monkey-patched)
            tgt_layer.load_state_dict(src_layer.state_dict(), strict=False)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        labels=None,
        **kwargs
    ):
        # If decoding normally, use parent forward
        if decoder_input_ids is None:
            return super().forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                labels=labels,
                **kwargs
            )

        # Encode
        encoder_outputs = self.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        encoder_hidden_states = encoder_outputs.last_hidden_state

        # Decode with wait-k (handled in custom decoder layers)
        decoder_outputs = self.model.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            return_dict=True,
        )

        sequence_output = decoder_outputs.last_hidden_state
        logits = self.lm_head(sequence_output)

        # Loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                logits.view(-1, self.config.vocab_size),
                labels.view(-1)
            )

        return {'loss': loss, 'logits': logits}


### 1. Load and preprocess data

In [57]:
from datasets import load_dataset, DatasetDict

/Users/eileen/contextual-live-translation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [58]:
opensub = load_dataset("opus100", "en-fr")
iwslt = load_dataset("IWSLT/iwslt2017", "iwslt2017-en-fr")

print(opensub)
print(iwslt)

Generating validation split: 100%|██████████| 890/890 [00:00<00:00, 114661.83 examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 232825
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 8597
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 890
    })
})


In [55]:
iwslt.save_to_disk("./data/iwslt")

Saving the dataset (1/1 shards): 100%|██████████| 890/890 [00:00<00:00, 405665.13 examples/s]


In [59]:
# choose split ratio
total = 300000
test_ratio = 0.045
val_ratio  = 0.01

test_size = int(total * test_ratio)
val_size  = int(total * val_ratio)
train_size = total - val_size - test_size

opensub_train = opensub["train"].select(range(0, train_size))
opensub_val   = opensub["train"].select(range(train_size, train_size + val_size))
opensub_test  = opensub["train"].select(range(train_size + val_size, total))

In [60]:
opensub_splits = DatasetDict({
    "train": opensub_train,
    "validation": opensub_val,
    "test": opensub_test
})

opensub_splits

DatasetDict({
    train: Dataset({
        features: ['id', 'meta', 'translation'],
        num_rows: 283500
    })
    validation: Dataset({
        features: ['id', 'meta', 'translation'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'meta', 'translation'],
        num_rows: 13500
    })
})

In [61]:
from datasets import concatenate_datasets
combined = DatasetDict({
    "train": concatenate_datasets([iwslt["train"], opensub_splits["train"]]),
    "train": concatenate_datasets([iwslt["train"], opensub_splits["train"]]),
    "validation": concatenate_datasets([iwslt["validation"], opensub_splits["validation"]]),
    "test": concatenate_datasets([iwslt["test"], opensub_splits["test"]])
})

combined

DatasetDict({
    train: Dataset({
        features: ['translation', 'id', 'meta'],
        num_rows: 516325
    })
    validation: Dataset({
        features: ['translation', 'id', 'meta'],
        num_rows: 3890
    })
    test: Dataset({
        features: ['translation', 'id', 'meta'],
        num_rows: 22097
    })
})

In [62]:
from langdetect import detect, LangDetectException
def to_df(ds, src_lang="en", tgt_lang="fr", dataset_name=""):
    data = []
    swap_count = 0
    skip_count = 0
    
    # First, check if we need to swap keys by analyzing multiple samples
    sample_size = min(20, len(ds))
    en_is_french = 0
    fr_is_french = 0
    
    for i in range(sample_size):
        row = ds[i]
        tr = row["translation"]
        if isinstance(tr, dict) and 'en' in tr and 'fr' in tr:
            try:
                en_detected = detect(tr['en'])
                fr_detected = detect(tr['fr'])
                if en_detected == 'fr':
                    en_is_french += 1
                if fr_detected == 'fr':
                    fr_is_french += 1
            except LangDetectException:
                pass
    
    # Determine if keys are swapped based on sample analysis
    keys_swapped = en_is_french > fr_is_french
    if keys_swapped:
        print(f"Note: Detected that OpenSubtitles dataset has swapped keys. 'en' contains French, 'fr' contains English.")
        print(f"  (Checked {sample_size} samples: {en_is_french} had French in 'en', {fr_is_french} had French in 'fr')")
    
    for row in ds:
        tr = row["translation"]
        # Handle different dataset structures
        # IWSLT: translation is a list of dicts
        if isinstance(tr, list):
            # For IWSLT, process each item in the list
            for item in tr:
                if isinstance(item, dict) and src_lang in item and tgt_lang in item:
                    data.append({"src": item[src_lang], "tgt": item[tgt_lang]})
        # OpenSubtitles: translation is a dict
        elif isinstance(tr, dict):
            if 'en' in tr and 'fr' in tr:
                # Swap if we determined keys are swapped
                if keys_swapped:
                    src_text = tr['fr']  # 'fr' key actually has English
                    tgt_text = tr['en']  # 'en' key actually has French
                else:
                    src_text = tr['en']
                    tgt_text = tr['fr']
                
                # Skip if both appear to be English (data quality issue)
                try:
                    src_detected = detect(src_text)
                    tgt_detected = detect(tgt_text)
                    if src_detected == 'en' and tgt_detected == 'en':
                        skip_count += 1
                        continue  # Skip this row
                except LangDetectException:
                    pass  # Can't detect, include anyway
                
                data.append({"src": src_text, "tgt": tgt_text})
            elif src_lang in tr and tgt_lang in tr:
                # Fallback to original keys
                data.append({"src": tr[src_lang], "tgt": tr[tgt_lang]})
            else:
                # Debug: print what keys we actually have (only first time)
                if len(data) == 0:
                    print(f"Warning: Could not find expected keys. Available keys: {list(tr.keys())}")
    
    if skip_count > 0:
        print(f"Note: Skipped {skip_count} rows where both texts appeared to be English")
    
    return pd.DataFrame(data)

In [63]:
df_train = to_df(combined["train"], dataset_name="Combined")
df_val = to_df(combined["validation"], dataset_name="Combined")
df_test = to_df(combined["test"], dataset_name="Combined")


Note: Skipped 5943 rows where both texts appeared to be English
Note: Skipped 42 rows where both texts appeared to be English
Note: Skipped 208 rows where both texts appeared to be English


In [57]:
df_test.to_csv("df_test.csv", index=False)

In [65]:
def add_prev_tgt(df):
    """
    Create a dataframe with columns: prev_tgt, src, tgt
    where prev_tgt is the previous utterance in the target language.
    
    Args:
        df: DataFrame with 'src' and 'tgt' columns
        
    Returns:
        DataFrame with 'prev_tgt', 'src', 'tgt' columns
    """
    # Create a copy to avoid modifying the original
    result_df = df.copy()
    
    # Shift the 'tgt' column by 1 to get the previous target
    # This will make prev_tgt[i] = tgt[i-1]
    result_df['prev_tgt'] = result_df['tgt'].shift(1)
    
    # Reorder columns: prev_tgt, src, tgt
    result_df = result_df[['prev_tgt', 'src', 'tgt']]
    
    return result_df

# Example usage:
df_train_with_prev = add_prev_tgt(df_train)
df_val_with_prev = add_prev_tgt(df_val)
df_test_with_prev = add_prev_tgt(df_test)


In [78]:
# df_train_with_prev.to_csv("df_train.csv", index=False)
# df_val_with_prev.to_csv("df_val.csv", index=False)
df_test_with_prev.to_csv("df_test.csv", index=False)


In [16]:
df_train_with_prev = pd.read_csv("df_train.csv")
df_val_with_prev = pd.read_csv("df_val.csv")
df_test = pd.read_csv("df_test.csv")

In [67]:
train_ds = Dataset.from_pandas(df_train_with_prev)
val_ds = Dataset.from_pandas(df_val_with_prev)

final_ds = DatasetDict({
    "train": train_ds,
    "val": val_ds
})
final_ds


DatasetDict({
    train: Dataset({
        features: ['prev_tgt', 'src', 'tgt'],
        num_rows: 510382
    })
    val: Dataset({
        features: ['prev_tgt', 'src', 'tgt'],
        num_rows: 3848
    })
})

In [77]:
final_ds.save_to_disk("./data/final_ds")

Saving the dataset (1/1 shards): 100%|██████████| 3848/3848 [00:00<00:00, 367663.26 examples/s]


In [68]:
def preprocess(examples):
    # Handle batched inputs - each field is a list
    # Handle None/NaN values for prev_tgt (use empty string if None)
    prev_tgts = [str(prev) if prev is not None and str(prev) != 'nan' else "" 
                 for prev in examples["prev_tgt"]]
    
    # Create input strings by concatenating prev_tgt, separator, and src
    input_texts = [
        (prev_tgt + " </ctx> " + src).strip() if prev_tgt else src
        for prev_tgt, src in zip(prev_tgts, examples["src"])
    ]
    
    # Tokenize inputs
    inputs = tokenizer(
        input_texts,
        max_length=256,
        truncation=True,
        padding=True
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["tgt"],
            max_length=256,
            truncation=True,
            padding=True
        )["input_ids"]

    # Replace pad tokens with -100 so they are ignored by the loss
    labels = [
        [(lbl if lbl != tokenizer.pad_token_id else -100) for lbl in label_seq]
        for label_seq in labels
    ]

    inputs["labels"] = labels
    return inputs

In [69]:
from transformers import MarianConfig, MarianTokenizer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Create MarianWaitK model instead of using pretrained
# Load config from pretrained model
base_model_name = "Helsinki-NLP/opus-mt-en-fr"
config = MarianConfig.from_pretrained(base_model_name)

# Create tokenizer and add special token
tokenizer = MarianTokenizer.from_pretrained(base_model_name)
special_tokens = {"additional_special_tokens": ["</ctx>"]}
tokenizer.add_special_tokens(special_tokens)

# Create MarianWaitK model with k=3 (or your desired k value)
# Create pretrained model first to get weights
pretrained_model = MarianMTModel.from_pretrained(base_model_name)

# Create MarianWaitK model with k=3 (or your desired k value)
model = MarianWaitK(config, k=3)

# Copy pretrained weights from base model (preserves wait-k mechanism)
model.copy_decoder_weights_from(pretrained_model)
# Copy encoder weights
model.model.encoder.load_state_dict(pretrained_model.model.encoder.state_dict())
# Copy decoder embedding and other components
model.model.decoder.embed_tokens.load_state_dict(pretrained_model.model.decoder.embed_tokens.state_dict())
model.model.decoder.embed_positions.load_state_dict(pretrained_model.model.decoder.embed_positions.state_dict())
if hasattr(pretrained_model.model.decoder, 'layer_norm'):
    model.model.decoder.layer_norm.load_state_dict(pretrained_model.model.decoder.layer_norm.state_dict())
# Copy language model head
model.lm_head.load_state_dict(pretrained_model.lm_head.state_dict())

# Resize token embeddings to account for new special token
model.resize_token_embeddings(len(tokenizer))

# Move to device
model = model.to(device)
model.train()


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


NameError: name 'MarianWaitK' is not defined

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from datasets import load_dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add context separator token
special_tokens = {"additional_special_tokens": ["</ctx>"]}
tokenizer.add_special_tokens(special_tokens)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# Move to device
model = model.to(device)
model.train()


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'torch' is not defined

In [71]:
processed_ds = final_ds.map(preprocess, batched=True, remove_columns=final_ds["train"].column_names)

Map:   0%|          | 0/510382 [00:00<?, ? examples/s]/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 3848/3848 [00:00<00:00, 5829.56 examples/s]


In [74]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./model_out",
    per_device_train_batch_size=32,   # adjust for memory
    per_device_eval_batch_size=32,
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=100,
    run_name="waitk-contextual-translation",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=500,
    learning_rate=1e-5,
    num_train_epochs=10,
    fp16=False,             # DON'T use fp16 (unsupported on Mac GPUs)
    bf16=False,             # bf16 also unsupported
    fp16_opt_level="O0",
    eval_accumulation_steps=2,
    predict_with_generate=True,
    report_to="wandb",      # Enable wandb logging
)


In [75]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=processed_ds["train"],
    eval_dataset=processed_ds["val"]
)

In [76]:
trainer.train()

wandb: Currently logged in as: eileen-zhang (eileen-zhang-university-of-michigan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
50,1.415300,2.162316
100,1.281900,2.143297
150,1.281700,2.136147


RuntimeError: MPS backend out of memory (MPS allocated: 17.83 GiB, other allocations: 8.41 GiB, max allowed: 27.20 GiB). Tried to allocate 1017.10 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

### Evaluation

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "./model_out/checkpoint-41000"    # your output_dir from training

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model_testing = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True).to("mps")
model_testing.eval()

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

In [36]:
def translate_waitk(model, src_en, prev_fr="", k=3, max_len=80,  debug=False):
    # Tokenize the source sentence
    full_tokens = tokenizer.tokenize(src_en)
    generated_ids = []
    
    # Get decoder start token ID from model config
    decoder_start_token_id = model.config.decoder_start_token_id if hasattr(model.config, 'decoder_start_token_id') else None
    if decoder_start_token_id is None:
        decoder_start_token_id = tokenizer.pad_token_id

    for t in range(max_len):
        # reveal source according to wait-k rule
        visible_src = full_tokens[: min(len(full_tokens), k + t)]
        visible_src_text = tokenizer.convert_tokens_to_string(visible_src)

        # build the input text with context (encoder input)
        if prev_fr:
            encoder_input = prev_fr + " </ctx> " + visible_src_text
        else:
            encoder_input = visible_src_text

        # Tokenize encoder input
        enc = tokenizer(
            encoder_input, return_tensors="pt", truncation=True, max_length=512
        ).to("mps")
        
        # Prepare decoder input with previously generated tokens
        # For seq2seq, we need to prepend decoder_start_token_id if this is the first token
        if len(generated_ids) == 0:
            decoder_input_ids = torch.tensor([[decoder_start_token_id]], device="mps")
        else:
            # Include decoder_start_token_id at the beginning, then generated tokens
            decoder_input_ids = torch.tensor([[decoder_start_token_id] + generated_ids], device="mps")

        # Forward pass to get next token logits
        with torch.no_grad():
            outputs = model(
                input_ids=enc["input_ids"],
                attention_mask=enc.get("attention_mask", None),
                decoder_input_ids=decoder_input_ids
            )
            
            # Get logits for the last position
            next_token_logits = outputs.logits[0, -1, :]
            
            # Apply temperature or just take argmax
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            
            if debug and t < 10:  # Debug first few tokens
                token_text = tokenizer.decode([next_token_id], skip_special_tokens=True)
                top_k_tokens = torch.topk(next_token_logits, 3)
                top_tokens_str = ", ".join([f"{tokenizer.decode([idx.item()], skip_special_tokens=True)}({idx.item()})" 
                                           for idx in top_k_tokens.indices])
                print(f"Step {t}: token_id={next_token_id}, token='{token_text}', top3=[{top_tokens_str}], EOS={next_token_id == tokenizer.eos_token_id}, PAD={next_token_id == tokenizer.pad_token_id}")

        # Only stop on EOS, not PAD (PAD might be a valid token in some cases)
        # Also, don't stop immediately if we haven't generated at least a few tokens
        if next_token_id == tokenizer.eos_token_id:
            if debug:
                print(f"Stopping at step {t}: EOS token generated")
            break
        
        # Don't add PAD tokens to output
        if next_token_id == tokenizer.pad_token_id:
            if debug:
                print(f"Skipping PAD token at step {t}")
            continue

        generated_ids.append(next_token_id)

    if len(generated_ids) == 0:
        if debug:
            print("No tokens generated!")
        return ""
    
    result = tokenizer.decode(generated_ids, skip_special_tokens=True)
    if debug:
        print(f"Generated {len(generated_ids)} tokens, decoded: '{result}'")
    return result


In [45]:
src = "I was shopping for groceries with my husband"
ctx = "Je t'ai vu au marché hier."   # French previous sentence

translation = translate_waitk(model_testing,src, ctx, k=3,debug=True)

print(translation)

Step 0: token_id=131, token='Je', top3=[Je(131), J(234), -(41)], EOS=False, PAD=False
Step 1: token_id=25166, token='faisais', top3=[faisais(25166), suis(558), voulais(6633)], EOS=False, PAD=False
Step 2: token_id=13, token='des', top3=[des(13), du(22), les(16)], EOS=False, PAD=False
Step 3: token_id=4220, token='courses', top3=[courses(4220), emp(10993), shopping(6312)], EOS=False, PAD=False
Step 4: token_id=27, token='pour', top3=[pour(27), d(20), .(3)], EOS=False, PAD=False
Step 5: token_id=13, token='des', top3=[des(13), l(14), les(16)], EOS=False, PAD=False
Step 6: token_id=4220, token='courses', top3=[courses(4220), épi(13135), (49)], EOS=False, PAD=False
Step 7: token_id=72, token='avec', top3=[avec(72), chez(845), auprès(1311)], EOS=False, PAD=False
Step 8: token_id=505, token='mon', top3=[mon(505), son(113), ma(589)], EOS=False, PAD=False
Step 9: token_id=5961, token='mari', top3=[mari(5961), époux(20965), père(2557)], EOS=False, PAD=False
Stopping at step 11: EOS token genera

### Test Set Evaluation (BLEU, chrF, COMET, Latency)


In [ ]:
# Install evaluation packages if needed
# !pip install sacrebleu unbabel-comet


In [2]:
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
import time
import numpy as np
from tqdm.auto import tqdm

# Try to import COMET (optional, may require GPU for best performance)
try:
    from comet import download_model, load_from_checkpoint
    COMET_AVAILABLE = True
    print("COMET is available")
except ImportError:
    COMET_AVAILABLE = False
    print("COMET not available. Install with: pip install unbabel-comet")


/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


COMET is available


In [19]:
# Load test data
df_test = pd.read_csv("df_test.csv")
print(f"Test set size: {len(df_test)}")
print(df_test.head())


Test set size: 12996
                                            prev_tgt  \
0                                                NaN   
1  Il y a plusieurs années, ici à Ted, Peter Skil...   
2  Et l'idée est plutôt simple. Des équipes de qu...   
3          Le marshmallow doit être placé au sommet.   
4  Bien que cela semble vraiment simple, c'est en...   

                                                 src  \
0  Several years ago here at TED, Peter Skillman ...   
1  And the idea's pretty simple:  Teams of four h...   
2                  The marshmallow has to be on top.   
3  And, though it seems really simple, it's actua...   
4  And so, I thought this was an interesting idea...   

                                                 tgt  
0  Il y a plusieurs années, ici à Ted, Peter Skil...  
1  Et l'idée est plutôt simple. Des équipes de qu...  
2          Le marshmallow doit être placé au sommet.  
3  Bien que cela semble vraiment simple, c'est en...  
4  J'ai trouvé que c'était une

In [20]:
def translate_waitk_with_latency(model, src_en, prev_fr="", k=3, max_len=80):
    """
    Translate with wait-k policy and track latency metrics.
    Returns: (translation, latency_info)
    
    Latency metrics:
    - Average Lagging (AL): measures how many source tokens behind the policy is
    - Computation time: wall-clock time for translation
    """
    start_time = time.time()
    
    # Tokenize the source sentence
    full_tokens = tokenizer.tokenize(src_en)
    src_len = len(full_tokens)
    generated_ids = []
    
    # Track read/write operations for latency calculation
    read_positions = []  # Source positions read at each step
    
    # Get decoder start token ID from model config
    decoder_start_token_id = model.config.decoder_start_token_id if hasattr(model.config, 'decoder_start_token_id') else None
    if decoder_start_token_id is None:
        decoder_start_token_id = tokenizer.pad_token_id

    for t in range(max_len):
        # reveal source according to wait-k rule
        visible_count = min(len(full_tokens), k + t)
        visible_src = full_tokens[:visible_count]
        visible_src_text = tokenizer.convert_tokens_to_string(visible_src)
        
        # Track source position for latency
        read_positions.append(visible_count)

        # build the input text with context (encoder input)
        if prev_fr:
            encoder_input = prev_fr + " </ctx> " + visible_src_text
        else:
            encoder_input = visible_src_text

        # Tokenize encoder input
        enc = tokenizer(
            encoder_input, return_tensors="pt", truncation=True, max_length=512
        ).to("mps")
        
        # Prepare decoder input with previously generated tokens
        if len(generated_ids) == 0:
            decoder_input_ids = torch.tensor([[decoder_start_token_id]], device="mps")
        else:
            decoder_input_ids = torch.tensor([[decoder_start_token_id] + generated_ids], device="mps")

        # Forward pass to get next token logits
        with torch.no_grad():
            outputs = model(
                input_ids=enc["input_ids"],
                attention_mask=enc.get("attention_mask", None),
                decoder_input_ids=decoder_input_ids
            )
            
            next_token_logits = outputs.logits[0, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()

        # Stop on EOS
        if next_token_id == tokenizer.eos_token_id:
            break
        
        # Skip PAD tokens
        if next_token_id == tokenizer.pad_token_id:
            continue

        generated_ids.append(next_token_id)

    end_time = time.time()
    computation_time = end_time - start_time
    
    # Calculate Average Lagging (AL)
    # AL = (1/τ) * Σ(g(t) - (t-1)*|x|/|y|)
    # For wait-k: g(t) = min(k + t - 1, |x|)
    tgt_len = len(generated_ids)
    if tgt_len > 0 and src_len > 0:
        al_sum = 0
        for t in range(1, tgt_len + 1):
            g_t = min(k + t - 1, src_len)  # source position after writing token t
            ideal_t = (t - 1) * src_len / tgt_len  # ideal position for simultaneous
            al_sum += g_t - ideal_t
        average_lagging = al_sum / tgt_len
    else:
        average_lagging = 0
    
    result = tokenizer.decode(generated_ids, skip_special_tokens=True) if generated_ids else ""
    
    latency_info = {
        "computation_time": computation_time,
        "average_lagging": average_lagging,
        "src_tokens": src_len,
        "tgt_tokens": tgt_len,
        "k": k
    }
    
    return result, latency_info


In [36]:
def evaluate_model(model, df_test, k=3, max_samples=None, use_context=True):
    """
    Evaluate the model on test set.
    
    Args:
        model: The translation model
        df_test: DataFrame with 'src' and 'tgt' columns
        k: Wait-k parameter
        max_samples: Limit evaluation to first N samples (for testing)
        use_context: Whether to use previous target as context
    
    Returns:
        Dictionary with evaluation metrics
    """
    if max_samples:
        df_eval = df_test.head(max_samples).copy()
    else:
        df_eval = df_test.copy()
    
    hypotheses = []
    references = []
    sources = []
    latency_metrics = []
    
    # Add prev_tgt column if using context
    if use_context:
        df_eval['prev_tgt'] = df_eval['tgt'].shift(1).fillna("")
    
    print(f"Evaluating {len(df_eval)} samples with k={k}...")
    
    for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
        src = str(row['src'])
        ref = str(row['tgt'])
        prev_tgt = str(row.get('prev_tgt', '')) if use_context else ""
        
        # Skip empty sources
        if not src or src == 'nan':
            continue
            
        # Translate with latency tracking
        hyp, latency_info = translate_waitk_with_latency(
            model, src, prev_fr=prev_tgt, k=k
        )
        
        hypotheses.append(hyp)
        references.append(ref)
        sources.append(src)
        latency_metrics.append(latency_info)
    
    print(f"\nGenerated {len(hypotheses)} translations")

    return hypotheses, references, sources, latency_metrics

def calculate_metrics(hypotheses, references, sources, latency_metrics, k):
    
    # Calculate BLEU
    bleu = BLEU()
    bleu_score = bleu.corpus_score(hypotheses, [references])
    print(f"\nBLEU Score: {bleu_score}")
    
    # Calculate chrF
    chrf = CHRF()
    chrf_score = chrf.corpus_score(hypotheses, [references])
    print(f"chrF Score: {chrf_score}")
    
    # Calculate latency metrics
    avg_computation_time = np.mean([m['computation_time'] for m in latency_metrics])
    avg_lagging = np.mean([m['average_lagging'] for m in latency_metrics])
    avg_src_tokens = np.mean([m['src_tokens'] for m in latency_metrics])
    avg_tgt_tokens = np.mean([m['tgt_tokens'] for m in latency_metrics])
    
    print(f"\nLatency Metrics (k={k}):")
    print(f"  Average Lagging (AL): {avg_lagging:.2f} tokens")
    print(f"  Average Computation Time: {avg_computation_time*1000:.2f} ms/sentence")
    print(f"  Average Source Tokens: {avg_src_tokens:.1f}")
    print(f"  Average Target Tokens: {avg_tgt_tokens:.1f}")
    
    results = {
        "bleu": bleu_score.score,
        "chrf": chrf_score.score,
        "avg_lagging": avg_lagging,
        "avg_computation_time_ms": avg_computation_time * 1000,
        "hypotheses": hypotheses,
        "references": references,
        "sources": sources,
        "latency_metrics": latency_metrics
    }
    
    # Calculate COMET if available
    if COMET_AVAILABLE:
        print("\nCalculating COMET score...")
        try:
            model_path = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(model_path)
            
            comet_data = [
                {"src": s, "mt": h, "ref": r}
                for s, h, r in zip(sources, hypotheses, references)
            ]
            
            # Use MPS GPU for faster COMET evaluation
            comet_output = comet_model.predict(
                comet_data, 
                batch_size=8, 
                accelerator="mps",  # Use Apple Silicon GPU
                num_workers=1  # Must be >= 1 to work with multiprocessing_context
            )
            comet_score = comet_output.system_score
            print(f"COMET Score: {comet_score:.4f}")
            results["comet"] = comet_score
        except Exception as e:
            print(f"COMET calculation failed: {e}")
            import traceback
            traceback.print_exc()
            results["comet"] = None
    else:
        results["comet"] = None
    
    return results


In [37]:
# Quick test with a small sample first
print("=" * 60)
print("QUICK TEST (100 samples)")
print("=" * 60)

h, r, s, l = evaluate_model(
    model_testing, 
    df_test, 
    k=3, 
    max_samples=30,
    use_context=True
)

results_quick = calculate_metrics(h, r, s, l, 3)

print("\n" + "=" * 60)
print("Quick Test Summary:")
print(f"  BLEU:  {results_quick['bleu']:.2f}")
print(f"  chrF:  {results_quick['chrf']:.2f}")
print(f"  COMET: {results_quick['comet']:.4f}" if results_quick['comet'] else "  COMET: N/A")
print(f"  Avg Lagging: {results_quick['avg_lagging']:.2f} tokens")
print(f"  Avg Time: {results_quick['avg_computation_time_ms']:.2f} ms/sentence")
print("=" * 60)


QUICK TEST (100 samples)
Evaluating 30 samples with k=3...


100%|██████████| 30/30 [00:12<00:00,  2.44it/s]



Generated 30 translations

BLEU Score: BLEU = 28.17 61.8/35.7/21.7/14.8 (BP = 0.971 ratio = 0.972 hyp_len = 651 ref_len = 670)
chrF Score: chrF2 = 53.77

Latency Metrics (k=3):
  Average Lagging (AL): 3.94 tokens
  Average Computation Time: 408.64 ms/sentence
  Average Source Tokens: 23.0
  Average Target Tokens: 26.9

Calculating COMET score...


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 56833.39it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 4/4 [00:05<00:00,  1.34s

COMET Score: 0.7306

Quick Test Summary:
  BLEU:  28.17
  chrF:  53.77
  COMET: 0.7306
  Avg Lagging: 3.94 tokens
  Avg Time: 408.64 ms/sentence


In [38]:
# Full evaluation on test set (can take a while)
print("=" * 60)
print("FULL EVALUATION")
print("=" * 60)

h_full, r_full, s_full, l_full = evaluate_model(
    model_testing, 
    df_test, 
    k=3, 
    max_samples=None,  # Use all samples
    use_context=True
)


FULL EVALUATION
Evaluating 12996 samples with k=3...


100%|██████████| 12996/12996 [58:23<00:00,  3.71it/s] 


Generated 12996 translations


In [39]:
results_full = calculate_metrics(h_full, r_full, s_full, l_full, 3)

print("\n" + "=" * 60)
print("Full Evaluation Summary:")
print(f"  BLEU:  {results_full['bleu']:.2f}")
print(f"  chrF:  {results_full['chrf']:.2f}")
print(f"  COMET: {results_full['comet']:.4f}" if results_full['comet'] else "  COMET: N/A")
print(f"  Avg Lagging: {results_full['avg_lagging']:.2f} tokens")
print(f"  Avg Time: {results_full['avg_computation_time_ms']:.2f} ms/sentence")
print("=" * 60)


BLEU Score: BLEU = 31.58 61.8/39.9/28.3/20.4 (BP = 0.914 ratio = 0.918 hyp_len = 188440 ref_len = 205287)
chrF Score: chrF2 = 53.60

Latency Metrics (k=3):
  Average Lagging (AL): 2.47 tokens
  Average Computation Time: 268.98 ms/sentence
  Average Source Tokens: 17.2
  Average Target Tokens: 17.5

Calculating COMET score...


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 58254.22it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1625/1625 [03:15<00:00, 

COMET Score: 0.7475

Full Evaluation Summary:
  BLEU:  31.58
  chrF:  53.60
  COMET: 0.7475
  Avg Lagging: 2.47 tokens
  Avg Time: 268.98 ms/sentence


In [ ]:
# Compare different k values (latency-quality tradeoff)
# Using a smaller sample for faster comparison
k_values = [1, 3, 5, 7, 10]
comparison_results = []

print("=" * 60)
print("LATENCY-QUALITY TRADEOFF ANALYSIS")
print("=" * 60)

for k in k_values:
    print(f"\n--- Evaluating k={k} ---")
    results_k = evaluate_model(
        model_testing, 
        df_test, 
        k=k, 
        max_samples=500,  # Use subset for faster comparison
        use_context=True
    )
    comparison_results.append({
        "k": k,
        "bleu": results_k['bleu'],
        "chrf": results_k['chrf'],
        "comet": results_k['comet'],
        "avg_lagging": results_k['avg_lagging'],
        "avg_time_ms": results_k['avg_computation_time_ms']
    })

# Display comparison table
print("\n" + "=" * 60)
print("COMPARISON TABLE")
print("=" * 60)
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv("waitk_comparison_results.csv", index=False)
print("\nResults saved to waitk_comparison_results.csv")


In [ ]:
# Visualize latency-quality tradeoff
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: BLEU vs Average Lagging
ax1 = axes[0]
ax1.plot(comparison_df['avg_lagging'], comparison_df['bleu'], 'bo-', markersize=10, linewidth=2)
for i, row in comparison_df.iterrows():
    ax1.annotate(f'k={row["k"]}', (row['avg_lagging'], row['bleu']), 
                 textcoords="offset points", xytext=(5, 5), fontsize=10)
ax1.set_xlabel('Average Lagging (tokens)', fontsize=12)
ax1.set_ylabel('BLEU Score', fontsize=12)
ax1.set_title('Quality vs Latency Tradeoff', fontsize=14)
ax1.grid(True, alpha=0.3)

# Plot 2: All metrics vs k
ax2 = axes[1]
ax2.plot(comparison_df['k'], comparison_df['bleu'], 'b-o', label='BLEU', markersize=8)
ax2.plot(comparison_df['k'], comparison_df['chrf'], 'g-s', label='chrF', markersize=8)
ax2.plot(comparison_df['k'], comparison_df['avg_lagging'], 'r-^', label='Avg Lagging', markersize=8)
ax2.set_xlabel('Wait-k Value', fontsize=12)
ax2.set_ylabel('Score / Tokens', fontsize=12)
ax2.set_title('Metrics vs Wait-k Parameter', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('waitk_tradeoff_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to waitk_tradeoff_analysis.png")


In [40]:
# Show example translations from evaluation
print("=" * 60)
print("EXAMPLE TRANSLATIONS")
print("=" * 60)

# Use results from the quick test
for i in range(min(10, len(results_quick['sources']))):
    print(f"\n--- Example {i+1} ---")
    print(f"Source (EN): {results_quick['sources'][i]}")
    print(f"Reference (FR): {results_quick['references'][i]}")
    print(f"Hypothesis (FR): {results_quick['hypotheses'][i]}")


EXAMPLE TRANSLATIONS

--- Example 1 ---
Source (EN): Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
Reference (FR): Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.
Hypothesis (FR): Il y a quelques années ici à TED, Peter Highbourman a présenté un défi de design appelé défi de la guimauve.

--- Example 2 ---
Source (EN): And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.
Reference (FR): Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow.
Hypothesis (FR): Et l'idée est assez simple: Des équipes de quatre doivent construire la plus haute free-standing structure sur 20 bâtons 

In [41]:
# Final Summary - save all results
final_results = {
    "model_checkpoint": "./model_out/checkpoint-66528",
    "test_set_size": len(df_test),
    "quick_test": {
        "samples": 100,
        "bleu": results_quick['bleu'],
        "chrf": results_quick['chrf'],
        "comet": results_quick['comet'],
        "avg_lagging": results_quick['avg_lagging'],
        "avg_time_ms": results_quick['avg_computation_time_ms']
    }
}

# Add full results if available
if 'results_full' in dir():
    final_results["full_evaluation"] = {
        "samples": len(df_test),
        "bleu": results_full['bleu'],
        "chrf": results_full['chrf'],
        "comet": results_full['comet'],
        "avg_lagging": results_full['avg_lagging'],
        "avg_time_ms": results_full['avg_computation_time_ms']
    }

import json
with open("evaluation_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("All results saved to evaluation_results.json")
print("\nFinal Results:")
print(json.dumps(final_results, indent=2))


All results saved to evaluation_results.json

Final Results:
{
  "model_checkpoint": "./model_out/checkpoint-66528",
  "test_set_size": 12996,
  "quick_test": {
    "samples": 100,
    "bleu": 28.166687684125613,
    "chrf": 53.76978287183002,
    "comet": 0.7305500527222951,
    "avg_lagging": 3.9405176615004875,
    "avg_time_ms": 408.6400349934896
  },
  "full_evaluation": {
    "samples": 12996,
    "bleu": 31.582655950853823,
    "chrf": 53.59584270657534,
    "comet": 0.7474701631104499,
    "avg_lagging": 2.4677781304939743,
    "avg_time_ms": 268.9795577735232
  }
}


In [50]:
example_trs, latency_info = translate_waitk_with_latency(
    model_testing, 
    "Can you take me to the Eiffel Tower?", 
    prev_fr="Où est-ce que vous devez aller ?", 
    k=3, 
    max_len=80
)

print(example_trs)

Pouvez-vous m'emmener à la Tour époustouflante ?
